In [2]:
import pandas as pd
from scipy import stats

match_pairs = pd.read_csv("../data/processed/match_pairs.csv")

rate_cols = ["ace_rate", "serve_error_rate",
             "positive_reception_rate", "perfect_reception_rate", "reception_error_rate",
             "attack_kill_rate", "attack_error_rate", "attack_blocked_rate", "attack_efficiency",
             "blocks_per_set"]

In [3]:
stats.ttest_rel(match_pairs["attack_efficiency_w"], match_pairs["attack_efficiency_l"])

TtestResult(statistic=np.float64(47.233524714527356), pvalue=np.float64(3.21606096042385e-310), df=np.int64(1674))

## Paired t-test
- A paired t-test compares the winner and the loser of the same match, so match conditions (league, season, opponent) are held constant.
- Attack efficiency: t = 47.2, p < 0.001, so the difference is clearly significant.

In [6]:
cohens_d = match_pairs["attack_efficiency_diff"].mean() / match_pairs["attack_efficiency_diff"].std()
print(d)

1.1540986938727233


## Effect size
- With 1,675 matches, even tiny differences become significant, so we also report the effect size (Cohen's d = mean difference / standard deviation of the differences).
- Attack efficiency: d = 1.15 (large effect). The winner had the higher attack efficiency in 89% of matches.
- Interpretation: d ≥ 0.2 small, ≥ 0.5 medium, ≥ 0.8 large.

In [10]:
rows = []

for gender in ["Men", "Women"]:
    league = match_pairs[match_pairs["league_gender"] == gender]
    for col in rate_cols:
        diff = league[col + "_diff"]
        test = stats.ttest_rel(league[col + "_w"], league[col + "_l"])
        cohens_d = diff.mean() / diff.std()
        rows.append({
            "league": gender,
            "skill": col,
            "mean_diff": diff.mean(),
            "cohens_d": cohens_d,
            "p_value": test.pvalue,
        })

results = pd.DataFrame(rows)
results

,league,skill,mean_diff,cohens_d,p_value
0,Men,ace_rate,0.019310,0.435214,1.361443e-31
1,Men,serve_error_rate,-0.024607,-0.375202,2.187714e-24
2,Men,positive_reception_rate,0.059053,0.539723,9.570615e-46
3,Men,perfect_reception_rate,0.044539,0.480242,1.833708e-37
4,Men,reception_error_rate,-0.023602,-0.419271,1.320511e-29
5,Men,attack_kill_rate,0.087354,1.089733,3.363571e-136
6,Men,attack_error_rate,-0.021716,-0.546236,1.100017e-46
7,Men,attack_blocked_rate,-0.031256,-0.713017,1.907714e-72
8,Men,attack_efficiency,0.140326,1.205210,8.514347e-156
9,Men,blocks_per_set,0.829024,0.702796,8.621459e-71


The same test and effect size for all ten skill rates, separately for the men's and women's leagues.

In [11]:
results["p_bonferroni"] = (results["p_value"] * len(results)).clip(upper=1)

results["abs_d"] = results["cohens_d"].abs()
results = results.sort_values(["league", "abs_d"], ascending=[True, False])
results

,league,skill,mean_diff,cohens_d,p_value,p_bonferroni,abs_d
8,Men,attack_efficiency,0.140326,1.205210,8.514347e-156,1.702869e-154,1.205210
5,Men,attack_kill_rate,0.087354,1.089733,3.363571e-136,6.727142e-135,1.089733
7,Men,attack_blocked_rate,-0.031256,-0.713017,1.907714e-72,3.815429e-71,0.713017
9,Men,blocks_per_set,0.829024,0.702796,8.621459e-71,1.724292e-69,0.702796
6,Men,attack_error_rate,-0.021716,-0.546236,1.100017e-46,2.200034e-45,0.546236
2,Men,positive_reception_rate,0.059053,0.539723,9.570615e-46,1.914123e-44,0.539723
3,Men,perfect_reception_rate,0.044539,0.480242,1.833708e-37,3.667416e-36,0.480242
0,Men,ace_rate,0.019310,0.435214,1.361443e-31,2.722887e-30,0.435214
4,Men,reception_error_rate,-0.023602,-0.419271,1.320511e-29,2.641022e-28,0.419271
1,Men,serve_error_rate,-0.024607,-0.375202,2.187714e-24,4.375428e-23,0.375202


## Which skills separate winners from losers?
- All 20 winner–loser differences are significant, also after Bonferroni correction.
- Attacking has by far the largest effect in both leagues (d > 1).
- Blocking ranks second in both leagues.
- Reception matters more in the men's league (medium effect, d ≈ 0.5) than in the women's league (small effect, d ≈ 0.25–0.29).
- In the women's league, serving has a larger effect than reception.